# Feature Engineering Pipeline — AND-103 Task 5

Implements the 9-stage pipeline defined in `docs/feature_engineering_spec.md`.

**Output:** `data/feature_matrix.csv`  
**Target:** Binary classification — `0` = PASS, `1` = NOT PASS  
**Temporal gate:** All features use only data strictly prior to each inspection date.

In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

DATA = Path("../data")
print("Libraries loaded.")

Libraries loaded.


## Stage 1 — Load raw data

In [2]:
df_insp = pd.read_csv(
    DATA / "inspection.csv",
    usecols=["ElevatingDevicesNumber", "Latest_INSPECTION_Date", "InspectionOutcome"],
).fillna("")

df_orders = pd.read_csv(
    DATA / "order.csv",
    usecols=["ElevatingDevicesNumber", "DateofIssue", "RISKSCORE",
             "ComplianceDate", "StatusofInspectionOrder", "DIRECTIVE"],
).fillna("")

df_static = pd.read_csv(
    DATA / "merged_elevator_data.csv",
    usecols=["ElevatingDevicesNumber", "Device Type"],
).fillna("")

print(f"Loaded: {len(df_insp):,} inspections | {len(df_orders):,} orders | {len(df_static):,} static records")

Loaded: 143,181 inspections | 162,172 orders | 52,031 static records


## Stage 2 — Normalize date fields to ISO YYYY-MM-DD

In [3]:
def to_iso(series):
    dt = pd.to_datetime(series, errors="coerce")
    return dt.dt.strftime("%Y-%m-%d").where(dt.notna(), pd.NA)

df_insp["Latest_INSPECTION_Date"] = to_iso(df_insp["Latest_INSPECTION_Date"])
df_orders["DateofIssue"]   = to_iso(df_orders["DateofIssue"])
df_orders["ComplianceDate"] = to_iso(df_orders["ComplianceDate"])

bad_insp = df_insp["Latest_INSPECTION_Date"].isna().sum()
bad_ord  = df_orders["DateofIssue"].isna().sum()
print(f"Unparseable inspection dates: {bad_insp:,} (will be excluded from base table)")
print(f"Unparseable order DateofIssue: {bad_ord:,} (will be excluded from order features)")

Unparseable inspection dates: 0 (will be excluded from base table)
Unparseable order DateofIssue: 96,544 (will be excluded from order features)


## Stage 3 — Build base inspection table

In [4]:
# Step 1: Exclude rows with null/empty InspectionOutcome
base = df_insp[df_insp["InspectionOutcome"].astype(str).str.strip() != ""].copy()

# Step 2: Exclude rows with unparseable inspection dates
base = base[base["Latest_INSPECTION_Date"].notna()].copy()

# Step 3: Deduplicate — sort then keep first occurrence per (elevator, date)
base = base.sort_values(["ElevatingDevicesNumber", "Latest_INSPECTION_Date"])
base = base.drop_duplicates(
    subset=["ElevatingDevicesNumber", "Latest_INSPECTION_Date"], keep="first"
).copy()

# Step 4: Encode target — "Passed" -> 0, anything else (non-null) -> 1
# Actual data values are "Passed", "Follow up", "Fail" — use substring match for 'pass'
base["target"] = (
    ~base["InspectionOutcome"].str.lower().str.contains("pass", na=False)
).astype(int)

# Add datetime column for temporal comparisons
base["current_dt"] = pd.to_datetime(base["Latest_INSPECTION_Date"])

print(f"Base table: {len(base):,} rows")
print(f"Target distribution:\n{base['target'].value_counts().to_string()}")
print(f"Pass rate: {(base['target'] == 0).mean():.1%}")

Base table: 132,212 rows
Target distribution:
target
1    107041
0     25171
Pass rate: 19.0%


## Stage 4 — Prior-inspection features

Temporal gate: `Latest_INSPECTION_Date < current_inspection_date` (strict).

In [5]:
# Prepare full inspection history with datetime (includes null-outcome rows)
insp_hist = df_insp.copy()
insp_hist["insp_date_dt"] = pd.to_datetime(insp_hist["Latest_INSPECTION_Date"], errors="coerce")
insp_hist = insp_hist[insp_hist["insp_date_dt"].notna()].copy()
insp_hist["is_pass"] = (
    insp_hist["InspectionOutcome"].str.lower().str.contains("pass", na=False)
).astype(int)

# Cross-join: each base row × all inspection records for the same elevator
prior_insp = base[["ElevatingDevicesNumber", "Latest_INSPECTION_Date", "current_dt"]].merge(
    insp_hist[["ElevatingDevicesNumber", "insp_date_dt", "is_pass"]],
    on="ElevatingDevicesNumber",
    how="left",
)

# Apply temporal gate (strict less-than)
prior_insp = prior_insp[prior_insp["insp_date_dt"] < prior_insp["current_dt"]].copy()

# --- Vectorized aggregations ---
grp_keys = ["ElevatingDevicesNumber", "Latest_INSPECTION_Date"]

insp_agg = prior_insp.groupby(grp_keys).agg(
    prior_inspection_count=("insp_date_dt", "count"),
    prior_pass_count=("is_pass", "sum"),
    _last_insp_date=("insp_date_dt", "max"),
).reset_index()

insp_agg["prior_fail_count"] = insp_agg["prior_inspection_count"] - insp_agg["prior_pass_count"]
insp_agg["prior_pass_rate"]  = insp_agg["prior_pass_count"] / insp_agg["prior_inspection_count"]

# days_since_last_inspection
base_dt = base[grp_keys + ["current_dt"]]
insp_agg = insp_agg.merge(base_dt, on=grp_keys, how="left")
insp_agg["days_since_last_inspection"] = (
    insp_agg["current_dt"] - insp_agg["_last_insp_date"]
).dt.days.astype(float)

# prior_inspection_frequency: mean of consecutive diffs per (elevator, date) group
prior_insp_s = prior_insp.sort_values(grp_keys + ["insp_date_dt"]).copy()
prior_insp_s["date_diff"] = prior_insp_s.groupby(grp_keys)["insp_date_dt"].diff().dt.days

freq_agg = (
    prior_insp_s.groupby(grp_keys)["date_diff"]
    .mean()
    .reset_index(name="prior_inspection_frequency")
)

insp_agg = insp_agg.merge(freq_agg, on=grp_keys, how="left")

# Select only needed columns
insp_features = insp_agg[grp_keys + [
    "prior_inspection_count", "prior_pass_count", "prior_fail_count",
    "prior_pass_rate", "days_since_last_inspection", "prior_inspection_frequency",
]]

# Left join to base; fill edge cases for first inspections
base = base.merge(insp_features, on=grp_keys, how="left")
base["prior_inspection_count"] = base["prior_inspection_count"].fillna(0).astype(int)
base["prior_pass_count"]       = base["prior_pass_count"].fillna(0).astype(int)
base["prior_fail_count"]       = base["prior_fail_count"].fillna(0).astype(int)
base["prior_pass_rate"]        = base["prior_pass_rate"].fillna(0.0)
# days_since_last_inspection and prior_inspection_frequency remain NaN for first inspections

print(f"Stage 4 complete. Rows with no prior inspections: "
      f"{(base['prior_inspection_count'] == 0).sum():,}")

Stage 4 complete. Rows with no prior inspections: 40,954


## Stage 5 — Prior-order features

Temporal gate: `DateofIssue < current_inspection_date` (strict).

In [6]:
# Prepare orders with datetime columns
orders = df_orders.copy()
orders["issue_dt"]      = pd.to_datetime(orders["DateofIssue"], errors="coerce")
orders["compliance_dt"] = pd.to_datetime(orders["ComplianceDate"], errors="coerce")
orders = orders[orders["issue_dt"].notna()].copy()
orders["RISKSCORE"] = pd.to_numeric(orders["RISKSCORE"], errors="coerce")
orders["directive_val"] = orders["DIRECTIVE"].str.strip().replace("", pd.NA)

# Cross-join: each base row × all orders for the same elevator
prior_ord = base[grp_keys + ["current_dt"]].merge(
    orders[["ElevatingDevicesNumber", "issue_dt", "compliance_dt",
            "RISKSCORE", "StatusofInspectionOrder", "directive_val"]],
    on="ElevatingDevicesNumber",
    how="left",
)

# Apply temporal gate (strict less-than on DateofIssue)
prior_ord = prior_ord[prior_ord["issue_dt"] < prior_ord["current_dt"]].copy()

# Derived flags
prior_ord["is_overdue"] = (
    prior_ord["compliance_dt"].notna() &
    (prior_ord["compliance_dt"] < prior_ord["current_dt"])
).astype(int)

prior_ord["is_unresolved"] = (
    prior_ord["StatusofInspectionOrder"].str.strip().str.upper() != "RESOLVED"
).astype(int)

# Aggregations
order_agg = prior_ord.groupby(grp_keys).agg(
    prior_order_count=("issue_dt", "count"),
    max_prior_riskscore=("RISKSCORE", "max"),
    mean_prior_riskscore=("RISKSCORE", "mean"),
    prior_overdue_order_count=("is_overdue", "sum"),
    prior_unresolved_order_count=("is_unresolved", "sum"),
    distinct_directive_count=("directive_val", "nunique"),
).reset_index()

# Left join to base
base = base.merge(order_agg, on=grp_keys, how="left")

# Fill edge cases: no prior orders -> 0 for counts, NaN for stats
base["prior_order_count"]            = base["prior_order_count"].fillna(0).astype(int)
base["prior_overdue_order_count"]    = base["prior_overdue_order_count"].fillna(0).astype(int)
base["prior_unresolved_order_count"] = base["prior_unresolved_order_count"].fillna(0).astype(int)
base["distinct_directive_count"]     = base["distinct_directive_count"].fillna(0).astype(int)
# max_prior_riskscore and mean_prior_riskscore remain NaN when no prior orders

print(f"Stage 5 complete. Rows with no prior orders: "
      f"{(base['prior_order_count'] == 0).sum():,}")

Stage 5 complete. Rows with no prior orders: 81,063


## Stage 6 — Join static attributes (Device Type)

In [7]:
# Deduplicate static data to one row per elevator before joining
static_dedup = (
    df_static[["ElevatingDevicesNumber", "Device Type"]]
    .drop_duplicates(subset="ElevatingDevicesNumber", keep="first")
)

base = base.merge(static_dedup, on="ElevatingDevicesNumber", how="left")

no_match = base["Device Type"].isna().sum()
print(f"Stage 6 complete. Rows with no static match: {no_match:,}")

Stage 6 complete. Rows with no static match: 4,346


## Stage 7 — Encode categorical variables (Device Type)

## Stage 8 — Drop excluded columns

In [8]:
# Stage 7: Label-encode Device Type -> integer
# pd.Categorical.codes: -1 for NaN (numeric sentinel, acceptable per spec)
base["device_type_encoded"] = pd.Categorical(base["Device Type"]).codes

print("Device Type encoding:")
print(
    base[["Device Type", "device_type_encoded"]]
    .drop_duplicates()
    .sort_values("device_type_encoded")
    .to_string(index=False)
)

# Stage 8: Drop all columns not in the 16-column schema contract
SCHEMA_COLS = [
    "ElevatingDevicesNumber", "Latest_INSPECTION_Date", "target",
    "prior_inspection_count", "prior_fail_count", "prior_pass_count", "prior_pass_rate",
    "days_since_last_inspection", "prior_inspection_frequency",
    "prior_order_count", "max_prior_riskscore", "mean_prior_riskscore",
    "prior_overdue_order_count", "prior_unresolved_order_count", "distinct_directive_count",
    "device_type_encoded",
]

df_final = base[SCHEMA_COLS].copy()
print(f"\nFinal schema: {list(df_final.columns)}")

Device Type encoding:
         Device Type  device_type_encoded
                 NaN                   -1
    Freight Elevator                    0
  Freight Elevator-E                    1
  Freight Elevator-P                    2
       LULA Elevator                    3
 Material Lift - ATD                    4
Observation Elevator                    5
  Passenger Elevator                    6
   Sidewalk Elevator                    7
Special Installation                    8
  Temporary Elevator                    9

Final schema: ['ElevatingDevicesNumber', 'Latest_INSPECTION_Date', 'target', 'prior_inspection_count', 'prior_fail_count', 'prior_pass_count', 'prior_pass_rate', 'days_since_last_inspection', 'prior_inspection_frequency', 'prior_order_count', 'max_prior_riskscore', 'mean_prior_riskscore', 'prior_overdue_order_count', 'prior_unresolved_order_count', 'distinct_directive_count', 'device_type_encoded']


## Stage 9 — Save output + validation

In [9]:
# --- Validation block (spec §6.1–§6.3) ---
print(f"Row count:   {len(df_final):,}")
print(f"Column count: {len(df_final.columns)}")

# §6.2 — duplicate check
n_dupes = len(df_final) - df_final.drop_duplicates(
    subset=["ElevatingDevicesNumber", "Latest_INSPECTION_Date"]
).shape[0]
assert n_dupes == 0, f"Duplicate rows found: {n_dupes}"
print("No duplicate rows: OK")

# §6.3 — class presence
classes = set(df_final["target"].unique())
assert 0 in classes and 1 in classes, f"Missing target class. Found: {classes}"
class_dist = df_final["target"].value_counts(normalize=True)
print(f"Target class distribution:\n{class_dist.to_string()}")
for cls, pct in class_dist.items():
    if pct > 0.95:
        warnings.warn(f"Class imbalance: class {cls} = {pct:.1%} of rows (threshold 95%)")

# §6.5 — schema
assert list(df_final.columns) == SCHEMA_COLS or set(df_final.columns) == set(SCHEMA_COLS)
print(f"\nSchema contract: OK ({len(df_final.columns)} columns)")

# --- Save ---
out_path = DATA / "feature_matrix.csv"
df_final.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nSaved: {out_path.resolve()}")

Row count:   132,212
Column count: 16
No duplicate rows: OK
Target class distribution:
target
1    0.809616
0    0.190384

Schema contract: OK (16 columns)



Saved: C:\Users\juanjanica\Documents\Proyecto_CodeBoxx\rocket-elevators-ops-dashboard\rocket-elevators-ops-dashboard\data\feature_matrix.csv
